# 05 · What should we have ordered?

Start with one item and a decision we can check by hand. Then ask whether a method with a lower forecast loss produces cheaper stock at similar service. The earlier Mark III replay already showed that a cheaper path can lose more sales. Here we freeze rules on the policy-validation dates, preserve the receipt ledger, and read April separately.

The demand fed into this experiment is recorded M5 sales. The starting stock and lost units are simulated; the cost coefficients are scenarios.

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
root = next((place for place in (current, *current.parents) if (place / "configs/config.yaml").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook from the project root or notebooks directory.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.data_audit import load_config

config_path = root / "configs/config.yaml"
config = load_config(config_path)
forecast = root / config["forecast"]["output_dir"]
report = root / config["artifacts"]["report_dir"] / "inventory"
processed = root / config["data"]["processed_dir"]

from src.inventory_sim import opening_state, run_inventory, simulate

if not (report / "run_manifest.json").is_file():
    report = run_inventory(config_path)
policy = json.loads((report / "policy.json").read_text())
selected = pd.read_csv(report / "selected_policies.csv")
front = pd.read_csv(report / "base_frontiers.csv")
config_inv = config["inventory"]

## 1. Work a review through by hand

On Tuesday morning, eight units carried over and four confirmed units arrive. The buyer can use 12. With a one-day lead and no other receipts outstanding, the inventory position is also 12. The two-day target is 20, so the buyer orders eight. When demand arrives at 15 units, 12 are filled and three are lost. Those eight purchased units enter usable stock on Wednesday. The ledger below checks each slot.

In [ ]:
opening = opening_state([8], lead=1)
opening.inbound[0] = 4
demand, target = [[15], [5], [8]], [[20], [18], [12]]
total, closing, ledger = simulate(demand, target, opening, capture=True)
columns = ["start", "receipts", "available", "position", "target", "ordered", "demand", "filled", "lost", "ending", "pipeline"]
worked = pd.DataFrame([{key: row[key].item() for key in columns} for row in ledger],
                      index=pd.date_range("2024-01-02", periods=len(ledger)))
display(worked)
assert worked.iloc[0]["lost"] == 3
assert worked.iloc[1]["receipts"] == worked.iloc[0]["ordered"]
print(f"Three-day unit fill: {total['filled'].item() / total['demand'].item():.1%}")

If I put Tuesday's new order on the shelf before demand, Tuesday appears to have no loss. The test checks the order arrives on Wednesday. A separately tested seven-day purchase appears seven days later. Across the replay, demand equals fulfilled plus lost, inventory reconciles from receipts and fulfilled units, and the pipeline reconciles from orders and receipts.

## 2. See what a dollar of price does in the model

Take the median listed price from up to 13 completed weeks ending before policy validation. When an item has no such price, use the store's department median, then its store median, and record the source. Estimate acquisition value as price times one minus the margin fraction. Daily holding cost is acquisition value times annual carry divided by 365. The stockout penalty starts with the lost margin and applies a scenario multiplier. The optional fee is per positive SKU order line.

Use the newsvendor critical fractile as a policy candidate. Compare the one-day formula with the protection-horizon holding proxy. Since the two cost terms scale by price, price cancels from their ratio within a scenario. The item price still matters to the dollar cost. Both formulas simplify the repeated-review lost-sales problem. Inside the fitted quantile grid we interpolate. Outside it, we use the closest trained quantile and record the clipping.

In [ ]:
reference = pd.read_parquet(report / "reference_prices.parquet")
display(reference["price_source"].value_counts().to_frame("SKU series"))
base = next(item for item in policy["scenarios"] if item["name"] == "base")
worked_price = 4.0
display(pd.DataFrame([{
    "selling price": worked_price, "assumed acquisition value": worked_price * (1 - base["margin"]),
    "holding per unit-day": worked_price * base["holding_per_price"],
    "lost unit penalty": worked_price * base["shortage_per_price"],
}]).round(5))
levels = pd.read_csv(report / "critical_fractiles.csv")
display(levels.loc[levels["scenario"].eq("base")].round(5))

## 3. Separate the forecast from the way a buyer protects it

Cross each forecast with three rules. Use its calibrated predictive quantile, add its own nonnegative residual buffer to its point forecast, or add the moving-average buffer to every model's point forecast. Holding the buffer fixed helps isolate the point forecast. The LightGBM residual comparisons start with its uncalibrated median; the direct rule uses its calibrated quantiles. Baseline predictive quantiles themselves are empirical point-plus-residual forecasts.

Pick the lowest modeled cost subject to 95% fill on policy validation, separately by model and lead. If no candidate reaches the floor, keep the highest-fill fallback for diagnosis and flag it. We also look for options within 0.25 percentage points of the incumbent's validation fill when the incumbent reaches the floor. Both rules are fixed before April scoring. The April frontier illustrates the outcomes of settings declared in advance; it has no role in selection.

For April, compare two starts. The common-control replay gives each candidate the stock and pipeline created by the moving-average incumbent through March 31. The continuous replay gives each candidate the state it created during validation and warm-up.

In [ ]:
fig, axes = plt.subplots(1, len(config_inv["lead_times"]), figsize=(11, 4), squeeze=False, layout="constrained")
for lead, ax in zip(config_inv["lead_times"], axes[0], strict=True):
    sku_all = front.loc[front["lead"].eq(lead) & front["regime"].eq("All")]
    for phase, rows in sku_all.groupby("phase"):
        rows = rows.sort_values("fill_rate")
        ax.plot(rows["fill_rate"], rows["cost"], marker=".", label={"policy_validation": "Validation", "final_cutover": "April cutover"}.get(phase, phase))
    ax.axvline(config_inv["fill_floor"], linestyle=":", color="#4d5859", label="Fill floor")
    ax.set(title=f"{lead}-day lead", xlabel="Achieved unit fill", ylabel="Total modeled cost (USD)")
    ax.xaxis.set_major_formatter(PercentFormatter(1))
    ax.legend(fontsize=8)
plt.show()

primary = selected.loc[selected["scenario"].eq("base") & selected["regime"].eq("All")
                       & selected["selection_mode"].eq("fill_floor")]
holdout = primary.loc[primary["phase"].eq("final_cutover")]
show = ["lead", "model", "rule", "validation_feasible", "fill_rate", "phase_fill_floor_met",
        "cost", "mean_on_hand", "lost", "stockout_rate", "ending_stock", "ending_pipeline"]
display(holdout[show].sort_values(["lead", "cost"]).round(4))

### What if the rules share the same buffer?

Check the 95th-percentile settings for the two main comparators. The table reports a fixed shared buffer and each model's own buffer alongside each forecast's predictive quantile. This helps locate the source of a difference. Comparing two selected policies changes both the forecast and how it is converted to stock.

In [ ]:
trajectories = pd.read_parquet(report / "scenario_trajectories.parquet", filters=[
    ("phase", "==", "final_cutover"), ("scenario", "==", "base"), ("regime", "==", "All"), ("alpha", "==", .95)
])
ml = next(name for name in trajectories["model"].unique() if name.startswith("lightgbm"))
display(trajectories.loc[trajectories["model"].isin([ml, "moving_average"]),
    ["lead", "model", "rule", "fill_rate", "mean_on_hand", "cost"]].sort_values(["lead", "rule", "model"]).round(4))
keys = ["lead", "model"]
sensitivity = primary.loc[primary["phase"].isin(["final_cutover", "final_continuous"])]
display(sensitivity.pivot(index=keys, columns="phase", values=["fill_rate", "cost"]).round(4))

## 4. Repricing versus changing decisions

Repricing holds orders and service fixed. It answers how much the same trajectory would cost under different assumptions. Reselection picks a new candidate using policy validation for each modeled scenario. It can change April orders and service. The normalized holding-to-shortage ratios stress the ranking and do not represent dollar costs. Before looking at April, we probed both sides of positive estimated cost break-even points from validation.

In [ ]:
fixed = pd.read_csv(report / "fixed_path_repricing.csv")
break_even = pd.read_csv(report / "validation_break_even.csv")
display(break_even.dropna(subset=["positive_break_even"])[["lead", "model", "positive_break_even"]].round(4))
scenario_names = ["base", "stockout_multiplier_0.5", "positive_item_order_fee_0.05", "ratio_0.01", "ratio_0.1"]
columns = ["scenario", "lead", "model", "rule", "alpha", "fill_rate", "cost"]
def comparison_slice(frame):
    return frame.loc[frame["scenario"].isin(scenario_names) & frame["lead"].eq(7)
                     & frame["model"].eq(ml) & frame["regime"].eq("All")
                     & frame["phase"].eq("final_cutover"), columns]
reselected = selected.loc[selected["selection_mode"].eq("fill_floor")]
display(pd.concat([comparison_slice(fixed).assign(method="fixed path"),
                   comparison_slice(reselected).assign(method="reselected on validation")])
        .sort_values(["scenario", "method"]).round(4))

## 5. Does a lower forecast loss buy better decisions?

Compare the selected base-cost policies with the moving-average incumbent by regime. The forecast column uses 90th-percentile scaled pinball on the 17 complete common windows. The simulator uses 24 observed demand days. A negative cost difference with worse fill is a trade. A negative cost difference with higher fill dominates the incumbent in these simulated conditions. A service-matched claim requires the absolute fill difference to stay within the stated tolerance; April service can depart from the validated target.

The default CA_1 cutover gives LightGBM lower modeled cost and higher fill than the moving average at both lead times. The selected rules differ, so the paired comparison describes whole forecasting-and-policy systems. The seven-day naïve rule is an example where a worse 90th-percentile forecast score can coexist with cheaper April stocking than the incumbent. These are short-window results under assumed costs and demand.

In the default run the moving-average candidates never reached the 95% validation floor for either lead, so the comparison has no matched-to-moving-average validation case. Under the fixed shared buffer the seven-day LightGBM case is slightly more expensive and fills slightly less than the moving average. That reversal makes it hard to credit the direct-quantile result to forecast choice alone.

In [ ]:
pairs = pd.read_csv(report / "forecast_inventory_pairs.csv")
pair_slice = pairs.loc[pairs["selection_mode"].eq("fill_floor") & pairs["model"].isin([ml, "naive"])]
display(pair_slice[["lead", "regime", "model", "forecast_origins", "q90_spl_delta", "cost_delta",
                    "fill_delta", "comparable_service"]].sort_values(["lead", "model", "regime"]).round(4))
matched = pairs.loc[pairs["selection_mode"].eq("matched_validation_service") & pairs["regime"].eq("All")]
display(matched[["lead", "model", "fill_delta", "comparable_service", "cost_delta"]].round(4))
print(f"Service-matched candidates against the incumbent on validation: {len(matched)}.")